# HW1 - Deep-RL
## MohammadParsa Dini
### Student ID: 21385609


In [2]:
!pip install gymnasium

## Installing Libs & visualisations

In [6]:
# coding: utf-8
import time
import numpy as np
import gymnasium as gym
from gymnasium.wrappers import TimeLimit
from gymnasium.envs.registration import register

# Create environment with rendering enabled
env = gym.make('FrozenLake-v1', map_name='4x4', is_slippery=True, render_mode='human')

# Reset environment
state, info = env.reset()

print("--- Data Types & Spaces ---")
print("Action Space:", env.action_space)       # Discrete(4) -> [Left, Down, Right, Up]
print("Observation Space:", env.observation_space) # Discrete(16) -> Grid cells 0 to 15
print("State Data Type:", type(state), "Value:", state) # Integer (0-15)
print("Info Dict:", info) # Probability of success/transition info

type(env.action_space)

--- Data Types & Spaces ---
Action Space: Discrete(4)
Observation Space: Discrete(16)
State Data Type: <class 'int'> Value: 0
Info Dict: {'prob': 1}


gymnasium.spaces.discrete.Discrete

gymnasium.wrappers.common.TimeLimit

# Problem 3

In this section we implemented the needed functions:

In [16]:
# coding: utf-8
import time
import numpy as np
import gymnasium as gym
from gymnasium.wrappers import TimeLimit
from gymnasium.envs.registration import register


register(id='SlipperyFrozenLake-v1',
    entry_point='gymnasium.envs.toy_text.frozen_lake:FrozenLakeEnv',
    kwargs={'map_name': '4x4', 'is_slippery': True})
register(id='FrozenLake-v1',
    entry_point='gymnasium.envs.toy_text.frozen_lake:FrozenLakeEnv',
    kwargs={'map_name': '4x4', 'is_slippery': False})


# """
# For policy_evaluation, policy_improvement, policy_iteration and value_iteration,
# the parameters P, nS, nA, gamma are defined as follows:

# 	P: nested dictionary
# 		From gym.core.Environment
# 		For each pair of states in [1, nS] and actions in [1, nA], P[state][action] is a list of
# 		tuple of the form (p_trans, next_state, reward, terminal) where
# 			- p_trans: float
# 				the transition probability of transitioning from "state" to "next_state" with "action"
# 			- next_state: int
# 				denotes the state we transition to (in range [0, nS - 1])
# 			- reward: int
# 				either 0 or 1, the reward for transitioning from "state" to
# 				"next_state" with "action"
# 			- terminal: bool
# 			  True when "next_state" is a terminal state (hole or goal), False otherwise
# 	nS: int
# 		number of states in the environment
# 	nA: int
# 		number of actions in the environment
# 	gamma: float
# 		Discount factor. Number in range [0, 1)
# """
# ------------------------------------------------------------------------------------------
def policy_evaluation(P, nS, nA, policy, gamma=0.9, eps=1e-3):
    value_function = np.zeros(nS)

    while True:
        delta = 0
        for s in range(nS):
            v = value_function[s]
            a = policy[s]
            new_v = 0.0
            for p_trans, next_state, reward, terminal in P[s][a]:
                if terminal:
                    new_v += p_trans * reward
                else:
                    new_v += p_trans * (reward + gamma * value_function[next_state])
            value_function[s] = new_v
            delta = max(delta, abs(v - new_v))
        if delta < eps:
            break

    return value_function


def policy_improvement(P, nS, nA, value_from_policy, policy, gamma=0.9):
    new_policy = np.zeros(nS, dtype=int)

    for s in range(nS):
        q_values = np.zeros(nA)
        for a in range(nA):
            for p_trans, next_state, reward, terminal in P[s][a]:
                if terminal:
                    q_values[a] += p_trans * reward
                else:
                    q_values[a] += p_trans * (reward + gamma * value_from_policy[next_state])
        new_policy[s] = np.argmax(q_values)

    return new_policy


def policy_iteration(P, nS, nA, gamma=0.9, eps=10e-3):
    value_function = np.zeros(nS)
    policy = np.zeros(nS, dtype=int)

    while True:
        value_function = policy_evaluation(P, nS, nA, policy, gamma, eps)
        new_policy = policy_improvement(P, nS, nA, value_function, policy, gamma)
        if np.array_equal(new_policy, policy):
            break
        policy = new_policy

    return value_function, policy


def value_iteration(P, nS, nA, gamma=0.9, eps=1e-3):
    value_function = np.zeros(nS)
    policy = np.zeros(nS, dtype=int)

    while True:
        delta = 0
        for s in range(nS):
            v = value_function[s]
            q_values = np.zeros(nA)
            for a in range(nA):
                for p_trans, next_state, reward, terminal in P[s][a]:
                    if terminal:
                        q_values[a] += p_trans * reward
                    else:
                        q_values[a] += p_trans * (reward + gamma * value_function[next_state])
            value_function[s] = np.max(q_values)
            delta = max(delta, abs(v - value_function[s]))
        if delta < eps:
            break

    # Extract greedy policy
    for s in range(nS):
        q_values = np.zeros(nA)
        for a in range(nA):
            for p_trans, next_state, reward, terminal in P[s][a]:
                if terminal:
                    q_values[a] += p_trans * reward
                else:
                    q_values[a] += p_trans * (reward + gamma * value_function[next_state])
        policy[s] = np.argmax(q_values)

    return value_function, policy
#----------------------------------------------------------------------------------------

def render_single(env, policy, max_steps=100):
    """This function does not need to be modified.
    Renders policy once on environment. Watch your agent play!

    Parameters
    ----------
    env: gym.core.Environment
      Environment to play on. Must have nS, nA, and P as attributes.
    policy: np.array of shape [env.nS]
      The action to take at a given state
    """

    episode_reward = 0
    state, _ = env.reset()
    while True:
        env.render()
        time.sleep(0.25)
        action = policy[state]
        state, reward, terminal, truncated, _ = env.step(action)
        episode_reward += reward
        if terminal or truncated:
            break
    env.render()
    if not terminal:
        print("The agent didn't reach a terminal state in {} steps.".format(max_steps))
    else:
        print("Episode reward: %f" % episode_reward)


# 3.A Results for Policy iteration on `FrozenLake-v1`

In [17]:
if __name__ == "__main__":
    # Create the environment with max_episode_steps built-in
    env = gym.make("FrozenLake-v1", render_mode="human", max_episode_steps=100)
    # env = gym.make("SlipperyFrozenLake-v1", render_mode="human", max_episode_steps=100)

    # Use env.unwrapped to access model dynamics (P) and space attributes safely
    P = env.unwrapped.P
    nS, nA = env.unwrapped.observation_space.n, env.unwrapped.action_space.n

    print("\n" + "-" * 25 + "\nBeginning Policy Iteration\n" + "-" * 25)
    V_pi, p_pi = policy_iteration(P, nS, nA, gamma=0.9, eps=1e-3)
    print("Optimal Value Function (Policy Iteration):\n", V_pi)
    print("Optimal Policy (Policy Iteration):\n", p_pi)
    render_single(env, p_pi, 100)




-------------------------
Beginning Policy Iteration
-------------------------
Optimal Value Function (Policy Iteration):
 [0.59049 0.6561  0.729   0.6561  0.6561  0.      0.81    0.      0.729
 0.81    0.9     0.      0.      0.9     1.      0.     ]
Optimal Policy (Policy Iteration):
 [1 2 1 0 1 0 1 0 2 1 1 0 0 2 2 0]
Episode reward: 1.000000


# 3.B Results for Value iteration on `FrozenLake-v1`

In [12]:
if __name__ == "__main__":
    # Create the environment with max_episode_steps built-in
    env = gym.make("FrozenLake-v1", render_mode="human", max_episode_steps=100)
    # env = gym.make("SlipperyFrozenLake-v1", render_mode="human", max_episode_steps=100)

    # Use env.unwrapped to access model dynamics (P) and space attributes safely
    P = env.unwrapped.P
    nS, nA = env.unwrapped.observation_space.n, env.unwrapped.action_space.n

    print("\n" + "-" * 25 + "\nBeginning Value Iteration\n" + "-" * 25)
    V_vi, p_vi = value_iteration(P, nS, nA, gamma=0.9, eps=1e-3)
    print("Optimal Value Function (Value Iteration):\n", V_vi)
    print("Optimal Policy (Value Iteration):\n", p_vi)
    render_single(env, p_vi, 100)


-------------------------
Beginning Policy Iteration
-------------------------
Optimal Value Function (Policy Iteration):
 [0.06476289 0.05841569 0.07252933 0.05378964 0.08867795 0.
 0.11137269 0.         0.14325408 0.24628655 0.29886952 0.
 0.         0.37915415 0.63865133 0.        ]
Optimal Policy (Policy Iteration):
 [0 3 0 3 0 0 0 0 3 1 0 0 0 2 1 0]
Episode reward: 1.000000

-------------------------
Beginning Value Iteration
-------------------------
Optimal Value Function (Value Iteration):
 [0.06428821 0.05807365 0.07231299 0.05356057 0.08830336 0.
 0.11127288 0.         0.14298808 0.24613328 0.29877497 0.
 0.         0.37905097 0.63860174 0.        ]
Optimal Policy (Value Iteration):
 [0 3 0 3 0 0 0 0 3 1 0 0 0 2 1 0]
Episode reward: 1.000000


# 3.C Results for Value iteration & Policy iteration on `SlipperyFrozenLake-v1`

In [18]:
if __name__ == "__main__":
    # Create the environment with max_episode_steps built-in
    # env = gym.make("FrozenLake-v1", render_mode="human", max_episode_steps=100)
    env = gym.make("SlipperyFrozenLake-v1", render_mode="human", max_episode_steps=100)

    # Use env.unwrapped to access model dynamics (P) and space attributes safely
    P = env.unwrapped.P
    nS, nA = env.unwrapped.observation_space.n, env.unwrapped.action_space.n

    print("\n" + "-" * 25 + "\nBeginning Policy Iteration\n" + "-" * 25)
    V_pi, p_pi = policy_iteration(P, nS, nA, gamma=0.9, eps=1e-3)
    print("Optimal Value Function (Policy Iteration):\n", V_pi)
    print("Optimal Policy (Policy Iteration):\n", p_pi)
    render_single(env, p_pi, 100)

    print("\n" + "-" * 25 + "\nBeginning Value Iteration\n" + "-" * 25)
    V_vi, p_vi = value_iteration(P, nS, nA, gamma=0.9, eps=1e-3)
    print("Optimal Value Function (Value Iteration):\n", V_vi)
    print("Optimal Policy (Value Iteration):\n", p_vi)
    render_single(env, p_vi, 100)


-------------------------
Beginning Policy Iteration
-------------------------
Optimal Value Function (Policy Iteration):
 [0.06476289 0.05841569 0.07252933 0.05378964 0.08867795 0.
 0.11137269 0.         0.14325408 0.24628655 0.29886952 0.
 0.         0.37915415 0.63865133 0.        ]
Optimal Policy (Policy Iteration):
 [0 3 0 3 0 0 0 0 3 1 0 0 0 2 1 0]
The agent didn't reach a terminal state in 100 steps.

-------------------------
Beginning Value Iteration
-------------------------
Optimal Value Function (Value Iteration):
 [0.06428821 0.05807365 0.07231299 0.05356057 0.08830336 0.
 0.11127288 0.         0.14298808 0.24613328 0.29877497 0.
 0.         0.37905097 0.63860174 0.        ]
Optimal Policy (Value Iteration):
 [0 3 0 3 0 0 0 0 3 1 0 0 0 2 1 0]
Episode reward: 1.000000
